In [ ]:
# ==============================================================================
# BƯỚC 3 v13-AttentionScore – Token Pruning theo Lassance et al. (2021)
# "A Study on Token Pruning for ColBERT" — arXiv:2112.06540
#
# Phương pháp: Attention Score (Eq. 2 trong paper)
#   i(t) = Σ_h Σ_j (A)_{h, t, j}
#   = với mỗi token t: tổng attention mà các token KHÁC "trả" cho t
#     (column-sum qua tất cả heads của attention matrix cuối)
#
# Khác biệt so với v12 (Ward hierarchical pooling):
#   - Không dùng Ward clustering
#   - Tính importance dựa thuần túy vào attention của QUERY encoder
#     (last layer, column-sum qua heads) — paper-correct
#   - Top-k ratio: giữ ratio*100% token có importance cao nhất
#   - Discarded tokens: DROPPED hoàn toàn (không pool, không cluster)
#     → đúng với paper gốc (paper prune hard, không soft-merge)
#
# Ablation axes (giống v12-Ablation):
#   - n_last_layers: [1, 2, 4, 8]   (paper dùng last layer → n=1 là paper-exact)
#   - scoring: traditional sum, importance-weighted sum
# ==============================================================================

print(">>> BƯỚC 3 v13-AttentionScore: Token Pruning (Lassance et al. 2021)")

import torch
import torch.nn.functional as F

# ==============================================================================
# CONFIG
# ==============================================================================
IMPORTANCE_VARIANT = 'attention_score_colbert_paper_v13'

# Top-k ratios: tỉ lệ token ĐƯỢC GIỮ LẠI (0.1 = giữ 10%, prune 90%)
TOPK_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Ablation: số layer cuối dùng để tính attention score
# n=1 → paper-exact (only last layer), n>1 → average/sum over last n layers
N_LAST_LAYERS_LIST = [1, 2, 4, 8]

# Scoring modes
SCORING_MODES = ['trad', 'weighted']

# ==============================================================================
# UTILS — tái sử dụng từ v12-Ablation, không thay đổi
# ==============================================================================
_cached_text_layers = None


def find_text_layers(model, force_rescan=False):
    global _cached_text_layers
    if _cached_text_layers is not None and not force_rescan:
        return _cached_text_layers
    candidates = []
    for name, module in model.named_modules():
        if not name.endswith('.layers') or not hasattr(module, '__len__'):
            continue
        is_vision = 'vision' in name.lower() or 'encoder' in name.lower()
        has_mlp   = hasattr(module[0], 'mlp') or hasattr(module[0], 'feed_forward')
        candidates.append({
            'name': name, 'module': module,
            'n_layers': len(module), 'is_vision': is_vision, 'has_mlp': has_mlp
        })
    text_c = [c for c in candidates if not c['is_vision'] and c['has_mlp']]
    best   = max(text_c or candidates, key=lambda c: c['n_layers'], default=None)
    _cached_text_layers = best['module'] if best else None
    return _cached_text_layers


def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    special_ids = set()
    tok = getattr(processor, 'tokenizer', processor)
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# ATTENTION COLLECTOR — hook-based, dùng khi model không return attentions
# ==============================================================================
class AttentionCollector:
    def __init__(self):
        self.attentions = []
        self.hooks      = []

    def _find_attn_module(self, layer):
        for attr in ['self_attn', 'attention', 'attn']:
            if hasattr(layer, attr):
                return getattr(layer, attr)
        return None

    def register_hooks(self, model, layer_indices):
        self.attentions.clear()
        self.remove_hooks()
        layers   = find_text_layers(model)
        if layers is None:
            raise RuntimeError("Cannot find transformer layers")
        n_layers = len(layers)
        for idx in layer_indices:
            idx = idx if idx >= 0 else n_layers + idx
            if not (0 <= idx < n_layers):
                continue
            attn_mod = self._find_attn_module(layers[idx])
            if attn_mod is None:
                continue

            def hook(module, inp, out):
                if isinstance(out, tuple) and len(out) >= 2:
                    attn = out[1]
                    if attn is not None:
                        self.attentions.append(attn.detach())
            self.hooks.append(attn_mod.register_forward_hook(hook))

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()

    def clear(self):
        self.attentions.clear()


# ==============================================================================
# ATTENTION SCORE — paper-correct implementation
#
# Eq. (2) trong Lassance et al. 2021:
#   i(t) = Σ_{h=0}^{H} Σ_{j=0}^{|t|} (A)_{h, t, j}
#
# Lưu ý quan trọng: paper tính COLUMN SUM — tức là tổng attention
# mà các token KHÁC dành cho token t.
# Về mặt tensor:
#   attention shape: (B, H, S_q, S_k)  (query × key)
#   (A)_{h, t, j}: head h, query position j "attends to" key position t
#   → column-sum: sum over j (query axis) → importance của key token t
#   → importance[t] = A[:, :, :, t].sum(dim=(1,2)) / n_heads
#
# Trong practice với self-attention S_q == S_k == S:
#   A shape: (B, H, S, S)
#   importance[b, t] = A[b, :, :, t].sum()   (sum over heads and query positions)
#
# Khi aggregate nhiều layers: average column-sum (uniform layer weight)
# vì paper dùng LAST layer (n=1), không có layer weighting.
# ==============================================================================

def compute_attention_score_importance(attn_list, content_mask, n_layers_used):
    """
    Tính importance score theo Eq. (2) của paper.

    Args:
        attn_list     : list of tensors, mỗi tensor shape (B, H, S, S)
                        Đây là attention weight của các last layers
        content_mask  : (B, S) float, 1 = content token, 0 = special/pad
        n_layers_used : int, số layer thực sự có trong attn_list

    Returns:
        importance    : (B, S) float, đã masked bởi content_mask
    """
    device = content_mask.device
    B, S   = content_mask.shape

    if not attn_list:
        # Fallback: uniform importance over content tokens
        return content_mask.clone()

    importance = torch.zeros(B, S, device=device, dtype=torch.float32)

    for attn in attn_list:
        attn = attn.float().to(device)            # (B, H, S_q, S_k)
        # Column-sum: tổng attention nhận vào mỗi token (key axis = dim=-1)
        # attn[:, :, :, t] = attention của tất cả queries đến key token t
        # sum over heads (dim=1) và query positions (dim=2)
        col_sum = attn.sum(dim=1).sum(dim=1)      # (B, S_k)
        importance = importance + col_sum

    # Average over layers
    importance = importance / max(n_layers_used, 1)

    # Mask ra special/pad tokens
    importance = importance * content_mask

    # Normalize về [0, 1] per sample (min-max) để stable
    # Note: paper không normalize, nhưng giúp stability khi dùng weighted scoring
    imp_max = importance.max(dim=-1, keepdim=True).values.clamp(min=1e-8)
    importance = importance / imp_max

    return importance                              # (B, S)


# ==============================================================================
# EXTRACTION FUNCTION — trả về (proj, importance)
#
# Chiến lược:
#   1. Thử output_attentions=True để lấy attention trực tiếp từ model output
#   2. Nếu không được, dùng AttentionCollector (forward hook)
#   3. Nếu cả hai fail, fallback về uniform importance
# ==============================================================================

def extract_embeddings_and_attn_importance(model, inputs, processor, n_last_layers=1):
    """
    Trả về:
        proj        : (B, S, D) normalized embeddings
        importance  : (B, S)   attention-score importance (paper Eq. 2)
        attn_ok     : bool     True nếu lấy được attention thực
    """
    collector = AttentionCollector()
    layer_indices = list(range(-n_last_layers, 0))  # e.g. [-1] for last layer

    # Register hooks as fallback
    try:
        collector.register_hooks(model, layer_indices)
    except Exception:
        pass

    try:
        with torch.no_grad():
            outputs = model(**inputs, output_attentions=True, return_dict=True)

            # Extract projection
            if hasattr(outputs, "last_hidden_state"):
                proj = outputs.last_hidden_state
            elif isinstance(outputs, torch.Tensor):
                proj = outputs
            else:
                raise RuntimeError("Unknown model output format")

            proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
            proj = proj * inputs["attention_mask"].unsqueeze(-1).float()

        content_mask = build_content_mask(inputs, processor).to(proj.device)

        # Prefer model-returned attentions
        if hasattr(outputs, "attentions") and outputs.attentions is not None:
            attn_list = list(outputs.attentions[-n_last_layers:])
        else:
            attn_list = collector.attentions[-n_last_layers:]

        n_actual   = len(attn_list)
        importance = compute_attention_score_importance(attn_list, content_mask, n_actual)
        attn_ok    = n_actual > 0

        collector.remove_hooks()
        collector.clear()
        return proj, importance, attn_ok

    except Exception as e:
        print(f"❌ Extraction error (n_last_layers={n_last_layers}): {e}")
        collector.remove_hooks()
        collector.clear()

        # Fallback: extract without attentions
        try:
            with torch.no_grad():
                outputs = model(**inputs, return_dict=True)
                proj = outputs.last_hidden_state
                proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
                proj = proj * inputs["attention_mask"].unsqueeze(-1).float()
        except Exception as e2:
            print(f"❌ Fallback extraction also failed: {e2}")
            B, S = inputs['input_ids'].shape
            D    = model.config.hidden_size
            proj = torch.zeros(B, S, D, device=inputs['input_ids'].device)

        content_mask = build_content_mask(inputs, processor).to(proj.device)
        importance   = content_mask.clone()        # uniform fallback
        return proj, importance, False


# ==============================================================================
# PRE-COMPUTED DOC MATRIX — tái dùng từ v12-Ablation
# ==============================================================================

def build_doc_matrix(docs_emb, device):
    n_docs  = len(docs_emb)
    max_len = max(d.shape[0] for d in docs_emb)
    D       = docs_emb[0].shape[1]
    doc_matrix = torch.zeros(n_docs, max_len, D,    device=device)
    doc_mask   = torch.zeros(n_docs, max_len,        device=device, dtype=torch.bool)
    for i, d in enumerate(docs_emb):
        L = d.shape[0]
        doc_matrix[i, :L] = F.normalize(d.float().to(device), dim=-1)
        doc_mask[i, :L]   = True
    return doc_matrix, doc_mask


# ==============================================================================
# FAST MAXSIM — batched einsum
# ==============================================================================

@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """
    Args:
        q_norm     : (K, D)  normalized query token vectors (K tokens kept)
        doc_matrix : (n_docs, max_len, D)
        doc_mask   : (n_docs, max_len) bool

    Returns:
        (K, n_docs) — per-token MaxSim scores
    """
    sim = torch.einsum('kd,nld->knl', q_norm, doc_matrix)     # (K, n_docs, max_len)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values                              # (K, n_docs)


# ==============================================================================
# ATTENTION-SCORE PRUNING SCORING
#
# Hard pruning: discarded tokens bị loại bỏ hoàn toàn (paper-correct).
# Hai scoring modes:
#   'trad'     : sum of MaxSim over kept tokens (như ColBERT gốc nhưng với K<T tokens)
#   'weighted' : importance-weighted sum of MaxSim
# ==============================================================================

def attention_prune_scores(q_embed, method_idx, importance_1d,
                           doc_matrix, doc_mask, topk_ratios, scoring='trad'):
    """
    Args:
        q_embed       : (S, D) full query embed (normalized)
        method_idx    : (M,)   content token indices
        importance_1d : (S,)   attention-score importance per query token
        doc_matrix    : (n_docs, max_len, D)
        doc_mask      : (n_docs, max_len) bool
        topk_ratios   : list[float]  ratios of tokens to KEEP
        scoring       : 'trad' | 'weighted'

    Returns:
        dict {ratio: scores (n_docs,)}
    """
    M      = method_idx.numel()
    n_docs = doc_matrix.shape[0]

    # Importance của content tokens only
    imp_content  = importance_1d[method_idx].float()   # (M,)
    q_vecs       = F.normalize(q_embed[method_idx].float(), dim=-1)   # (M, D)

    # Sort once descending — reuse across ratios
    sorted_idx   = torch.argsort(imp_content, descending=True)

    results = {}
    for ratio in topk_ratios:
        n_keep = max(1, round(M * ratio))
        kept   = sorted_idx[:n_keep]          # top-k importance indices

        q_kept  = q_vecs[kept]               # (K, D)
        M_kept  = fast_maxsim(q_kept, doc_matrix, doc_mask)   # (K, n_docs)

        if scoring == 'trad':
            # ColBERT-style sum (Eq. 1 in original ColBERT, restricted to K tokens)
            scores = M_kept.sum(dim=0)
        else:
            # Importance-weighted: each token's MaxSim is weighted by its importance
            imp_kept = imp_content[kept]     # (K,)
            w        = imp_kept / imp_kept.sum().clamp(min=1e-8)
            scores   = (M_kept * w.unsqueeze(-1)).sum(dim=0)

        results[ratio] = scores

    return results


print(">>> BƯỚC 3 v13-AttentionScore COMPLETE")


# ==============================================================================
# BƯỚC 3.5: UTILITY FUNCTIONS
# ==============================================================================
print(">>> BƯỚC 3.5: Loading utility functions...")

import numpy as np


def compute_ndcg(ranked_indices, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, idx in enumerate(ranked_indices[:k]) if idx in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0


def first_hit(top_k, gt_set):
    for r, idx in enumerate(top_k):
        if idx in gt_set:
            return r + 1
    return -1


def pretty_token(t):
    t = str(t).replace("\n", "\\n").replace(" ", "_")
    return t if len(t) <= 24 else (t[:22] + "..")


print("✅ Utility functions loaded")
print(">>> BƯỚC 3.5 COMPLETE")


# ==============================================================================
# BƯỚC 4: COMPREHENSIVE BATCH EVALUATION
# ==============================================================================
print(">>> BƯỚC 4: Process All Batches — Attention Score Pruning (Lassance et al. 2021)")
print(f"Processing {len(BATCH_RANGES)} document batches: {BATCH_RANGES}")
print(f"n_last_layers variants : {N_LAST_LAYERS_LIST}")
print(f"scoring modes          : {SCORING_MODES}")
print(f"TOPK_RATIOS            : {TOPK_RATIOS}")

import json
import os
import pickle
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR      = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
QUERY_BATCH_SIZE = 50


# ==============================================================================
# METHOD KEYS
# ==============================================================================
def make_method_keys():
    keys = ['traditional', 'trad_weighted']
    for n in N_LAST_LAYERS_LIST:
        for scoring in SCORING_MODES:
            for r in TOPK_RATIOS:
                keys.append(f"attn_L{n}_{scoring}_r{int(r*100)}")
    return keys

METHOD_KEYS = make_method_keys()


# ==============================================================================
# METRIC HELPERS
# ==============================================================================
def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}


def _add_metric(dst, src):
    dst['r1']    += int(src['r1'])
    dst['r5']    += int(src['r5'])
    dst['r10']   += int(src['r10'])
    dst['n1']    += float(src['n1'])
    dst['n5']    += float(src['n5'])
    dst['n10']   += float(src['n10'])
    dst['count'] += 1


def _ensure(store, key):
    if key not in store:
        store[key] = _init_metric()
    return store[key]


def hit_metrics(top10, gt):
    h = first_hit(top10, gt)
    return {
        'r1':  int(h != -1 and h <= 1),
        'r5':  int(h != -1 and h <= 5),
        'r10': int(h != -1 and h <= 10),
        'n1':  float(compute_ndcg(top10, gt, 1)),
        'n5':  float(compute_ndcg(top10, gt, 5)),
        'n10': float(compute_ndcg(top10, gt, 10)),
    }


def record(key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results  = []
all_token_results  = []
all_metrics        = {}
all_domain_metrics = {}
all_batch_stats    = []

print("\n" + "=" * 80)
print("BATCH LOOP PROCESSING")
print("=" * 80)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Processing Batch [{START_IDX}:{END_IDX}]")

    # =========================================================================
    # LOAD INDEX
    # =========================================================================
    index_path = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
    if not os.path.exists(index_path):
        print(f"  ⚠️  Index not found: {index_path}")
        continue

    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        if isinstance(saved, dict):
            fused_embeddings = saved.get('embeddings', [])
        else:
            fused_embeddings = saved
        print(f"  ✅ Loaded index: {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ Error loading index: {e}")
        continue

    # =========================================================================
    # LOAD BATCH DATA  (giống hệt v12-Ablation, không thay đổi logic)
    # =========================================================================
    print("  Loading batch data...")
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="    Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns:
                    cols.append('text_level')
                temp = temp[[c for c in cols if c in temp.columns]]
                dfs.append(temp)
            except Exception:
                pass

        if dfs:
            batch_df_enh = pd.concat(dfs, ignore_index=True)
            batch_df_enh = batch_df_enh.rename(columns={
                'vlm_text':   'vlm_text_enhanced',
                'text_level': 'text_level_enhanced'
            })
        else:
            batch_df_enh = pd.DataFrame()

        batch_df_final = pd.merge(
            batch_df_orig, batch_df_enh,
            on=['join_doc_name', 'layout_id'], how='left'
        )
        batch_df_final = batch_df_final.sort_values(by=['join_doc_name', 'page_id', 'layout_id'])

        is_header_type = batch_df_final['type'].isin(['title', 'section_header', 'header'])
        has_text_level = batch_df_final.get('text_level_enhanced',
                                            pd.Series(dtype=object)).notna()
        is_header = is_header_type | has_text_level
        batch_df_final['temp_header'] = batch_df_final['text'].where(is_header)
        batch_df_final['current_section'] = (
            batch_df_final.groupby('join_doc_name')['temp_header']
            .ffill().fillna("General Content")
        )

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        df = batch_df_final.copy()
        if 'img_enhanced_path' in df.columns:
            img_fnames = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p)))
            )
            df['img_data'] = img_fnames
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None
            df['img_type'] = None

        if 'image_binary' in df.columns:
            need_fallback = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[need_fallback, 'img_type'] = 'binary'
            df.loc[need_fallback, 'img_data'] = df.loc[need_fallback, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced', 'text', 'ocr_text', 'vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5:
                    return str(v)
            return "Document layout."

        raw_content = df.apply(_pick_text, axis=1)
        df['final_text'] = ("Section: " + df['current_section'].fillna('')
                            + " \n Content: " + raw_content)

        batch_sample_layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ Batch data: {len(batch_sample_layouts_df)} layouts")

    except Exception as e:
        print(f"  ❌ Error loading batch data: {e}")
        import traceback; traceback.print_exc()
        continue

    # =========================================================================
    # BUILD QA PAIRS
    # =========================================================================
    print("  Building QA pairs...")

    def calculate_iou(box1, box2):
        b1 = list(box1) if not isinstance(box1, list) else box1
        b2 = list(box2) if not isinstance(box2, list) else box2
        x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
        x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, x2 - x1) * max(0, y2 - y1)
        union  = ((b1[2]-b1[0])*(b1[3]-b1[1])) + ((b2[2]-b2[0])*(b2[3]-b2[1])) - inter
        return inter / union if union > 0 else 0.0

    batch_qa_pairs   = []
    batch_doc_lookup = {k: v for k, v in batch_sample_layouts_df.groupby('join_doc_name')}
    batch_avail_docs = set(batch_doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try:
                doc_data = json.loads(line)
            except Exception:
                continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in batch_avail_docs:
                continue
            doc_layouts    = batch_doc_lookup[target_doc]
            domain         = doc_data.get('domain', 'General')
            col_page       = 'page_idx' if 'page_idx' in doc_layouts.columns else 'page_id'
            current_doc_df = doc_layouts.copy()
            current_doc_df['safe_page'] = (
                pd.to_numeric(current_doc_df[col_page], errors='coerce')
                .fillna(-999).astype(int)
            )
            for q_item in doc_data.get('questions', []):
                gt_indices = []
                if 'layout_mapping' in q_item:
                    for target in q_item['layout_mapping']:
                        t_page, t_bbox = target['page'], target['bbox']
                        try:
                            t_page_int = int(t_page)
                            cands = pd.concat([
                                current_doc_df[current_doc_df['safe_page'] == t_page_int],
                                current_doc_df[current_doc_df['safe_page'] == t_page_int - 1],
                            ])
                            for idx, row in cands.iterrows():
                                if calculate_iou(row['bbox'], t_bbox) > 0.5:
                                    gt_indices.append(int(idx))
                        except Exception:
                            continue
                if gt_indices:
                    batch_qa_pairs.append({
                        'question':   q_item['Q'],
                        'gt_indices': list(set(gt_indices)),
                        'doc_name':   target_doc,
                        'domain':     domain,
                    })

    print(f"  ✅ Found {len(batch_qa_pairs)} QA pairs")

    if len(batch_qa_pairs) == 0:
        print("  ⚠️  No QA pairs, skipping")
        all_batch_stats.append({
            'batch': f"{START_IDX}-{END_IDX}",
            'n_layouts': len(fused_embeddings),
            'n_qa': 0, 'status': 'no_qa'
        })
        continue

    # =========================================================================
    # EVALUATE — ATTENTION SCORE PRUNING
    # =========================================================================
    print(f"  Evaluating {len(batch_qa_pairs)} queries across "
          f"{len(N_LAST_LAYERS_LIST)} n_last_layers × "
          f"{len(SCORING_MODES)} scoring × "
          f"{len(TOPK_RATIOS)} ratios ...")

    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs = len(fused_embeddings)
        total  = len(batch_qa_pairs)

        print("  Building doc matrix (once)...")
        doc_matrix, doc_mask = build_doc_matrix(fused_embeddings, device)
        print(f"  ✅ Doc matrix: {doc_matrix.shape}")

        batch_query_rows = []
        batch_token_rows = []

        for query_batch_start in range(0, total, QUERY_BATCH_SIZE):
            query_batch_end   = min(query_batch_start + QUERY_BATCH_SIZE, total)
            query_batch_range = batch_qa_pairs[query_batch_start:query_batch_end]

            pbar = tqdm(
                enumerate(query_batch_range),
                total=len(query_batch_range),
                desc=f"    Q[{query_batch_start}:{query_batch_end}]",
                leave=False
            )

            for local_q_idx, item in pbar:
                global_q_idx = query_batch_start + local_q_idx
                question     = item['question']
                gt_set       = set(item['gt_indices'])
                domain       = item['domain']

                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}

                with torch.no_grad():
                    q_inputs = processor.process_queries([question]).to(device)

                attn_mask_1d    = q_inputs['attention_mask'][0].float()
                content_mask_1d = build_content_mask(q_inputs, processor)[0].float()

                trad_idx   = torch.where(attn_mask_1d > 0)[0]
                method_idx = torch.where(content_mask_1d > 0)[0]
                if trad_idx.numel() == 0:
                    continue
                if method_idx.numel() == 0:
                    method_idx = trad_idx

                # -------------------------------------------------------
                # Extract embeddings & attention once per n_last_layers
                # (cache to avoid redundant forward passes)
                # -------------------------------------------------------
                embed_cache = {}   # n_layers -> (proj[0], importance[0])
                for n_layers in N_LAST_LAYERS_LIST:
                    proj, imp, attn_ok = extract_embeddings_and_attn_importance(
                        model, q_inputs, processor, n_last_layers=n_layers
                    )
                    embed_cache[n_layers] = (proj[0].float(), imp[0].float())

                # -------------------------------------------------------
                # BASELINE A: Traditional MaxSim (no pruning, all content tokens)
                # -------------------------------------------------------
                q_embed_base = embed_cache[N_LAST_LAYERS_LIST[0]][0]
                q_trad_norm  = F.normalize(q_embed_base[trad_idx].float(), dim=-1)
                M_trad       = fast_maxsim(q_trad_norm, doc_matrix, doc_mask)  # (M, n_docs)

                trad_scores = M_trad.sum(dim=0)
                trad_top10  = torch.topk(trad_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_m      = hit_metrics(trad_top10, gt_set)
                record('traditional', trad_m, domain)

                # BASELINE B: Traditional importance-weighted
                imp_base     = embed_cache[N_LAST_LAYERS_LIST[0]][1]
                imp_trad     = imp_base[trad_idx]
                w_trad       = imp_trad / imp_trad.sum().clamp(min=1e-8)
                trad_w_scores = (M_trad * w_trad.unsqueeze(-1)).sum(dim=0)
                trad_w_top10  = torch.topk(trad_w_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_w_m      = hit_metrics(trad_w_top10, gt_set)
                record('trad_weighted', trad_w_m, domain)

                query_row = {
                    'batch':         f"{START_IDX}-{END_IDX}",
                    'query_id':      global_q_idx,
                    'doc_name':      item['doc_name'],
                    'domain':        item['domain'],
                    'question':      question,
                    'trad_r@1':      trad_m['r1'],
                    'trad_r@5':      trad_m['r5'],
                    'trad_r@10':     trad_m['r10'],
                    'trad_ndcg@1':   round(trad_m['n1'],  4),
                    'trad_ndcg@5':   round(trad_m['n5'],  4),
                    'trad_ndcg@10':  round(trad_m['n10'], 4),
                    'tradW_r@1':     trad_w_m['r1'],
                    'tradW_r@5':     trad_w_m['r5'],
                    'tradW_r@10':    trad_w_m['r10'],
                    'tradW_ndcg@10': round(trad_w_m['n10'], 4),
                }

                # -------------------------------------------------------
                # ATTENTION SCORE PRUNING — across all (n_layers, scoring, ratio)
                # -------------------------------------------------------
                for n_layers in N_LAST_LAYERS_LIST:
                    q_embed, q_imp = embed_cache[n_layers]

                    for scoring in SCORING_MODES:
                        ratio_scores = attention_prune_scores(
                            q_embed, method_idx, q_imp,
                            doc_matrix, doc_mask,
                            topk_ratios=TOPK_RATIOS,
                            scoring=scoring,
                        )

                        for r in TOPK_RATIOS:
                            key    = f"attn_L{n_layers}_{scoring}_r{int(r*100)}"
                            scores = ratio_scores[r]
                            top10  = torch.topk(scores, k=min(10, n_docs)).indices.cpu().tolist()
                            m      = hit_metrics(top10, gt_set)
                            record(key, m, domain)
                            query_row.update({
                                f'{key}_r@1':     m['r1'],
                                f'{key}_r@5':     m['r5'],
                                f'{key}_r@10':    m['r10'],
                                f'{key}_ndcg@10': round(m['n10'], 4),
                            })

                batch_query_rows.append(query_row)

                # -------------------------------------------------------
                # Token-level analysis (reference: n=1, paper-exact layer)
                # -------------------------------------------------------
                q_imp_ref  = embed_cache[1][1]    # n_last_layers=1 importance
                q_emb_ref  = embed_cache[1][0]
                q_ref_norm = F.normalize(q_emb_ref[method_idx].float(), dim=-1)
                M_ref      = fast_maxsim(q_ref_norm, doc_matrix, doc_mask)

                method_np  = method_idx.cpu().numpy()
                imp_np     = q_imp_ref[method_idx].cpu().numpy()
                input_ids  = q_inputs['input_ids'][0].cpu().tolist()
                tok_strs   = processor.tokenizer.convert_ids_to_tokens(input_ids)
                best_scores = M_ref.max(dim=1).values.cpu().numpy()

                for local_t, global_t in enumerate(method_np):
                    tok_raw = (tok_strs[int(global_t)]
                               if int(global_t) < len(tok_strs) else "<UNK>")
                    batch_token_rows.append({
                        'batch':           f"{START_IDX}-{END_IDX}",
                        'query_id':        global_q_idx,
                        'token_idx':       int(global_t),
                        'token':           pretty_token(tok_raw),
                        'attn_importance': round(float(imp_np[local_t]), 8),
                        'maxsim_best':     round(float(best_scores[local_t]), 8),
                    })

            gc.collect()
            torch.cuda.empty_cache()
            print(f"      ✓ Q[{query_batch_start}:{query_batch_end}] done")

        all_query_results.extend(batch_query_rows)
        all_token_results.extend(batch_token_rows)

        # Batch summary
        t_m    = all_metrics.get('traditional', _init_metric())
        t_cnt  = t_m['count'] if t_m['count'] > 0 else 1
        t_r10  = t_m['r10'] / t_cnt * 100
        t_nd10 = t_m['n10'] / t_cnt
        print(f"  ✅ Batch done: {len(batch_query_rows)} Q, {len(batch_token_rows)} T")
        print(f"     Baseline R@10={t_r10:.1f}%  nDCG@10={t_nd10:.4f}")

        all_batch_stats.append({
            'batch':          f"{START_IDX}-{END_IDX}",
            'n_layouts':      len(fused_embeddings),
            'n_queries':      total,
            'trad_recall@10': round(t_r10, 2),
            'trad_ndcg@10':   round(t_nd10, 4),
            'status':         'ok'
        })

    except Exception as e:
        print(f"  ❌ Evaluation error: {e}")
        import traceback; traceback.print_exc()
        all_batch_stats.append({
            'batch': f"{START_IDX}-{END_IDX}",
            'status': 'error', 'error': str(e)
        })

    # =========================================================================
    # SAVE & CLEANUP
    # =========================================================================
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"),
                index=False
            )
        if batch_token_rows:
            pd.DataFrame(batch_token_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_tokens.csv"),
                index=False
            )
        print("  ✅ Batch files saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del doc_matrix, doc_mask
    del fused_embeddings, batch_sample_layouts_df, batch_qa_pairs
    del batch_df_orig, batch_df_enh, batch_df_final
    gc.collect()
    torch.cuda.empty_cache()
    print("  ✓ Batch memory cleaned\n")


# ==============================================================================
# FINAL: CONSOLIDATE & SUMMARY
# ==============================================================================
print("=" * 80)
print("CONSOLIDATING RESULTS")
print("=" * 80)

try:
    if all_query_results:
        df_all_q = pd.DataFrame(all_query_results)
        df_all_q.to_csv(os.path.join(WORKING_DIR, "MASTER_all_batches_queries.csv"), index=False)
        print(f"✅ {len(df_all_q)} queries → MASTER_all_batches_queries.csv")

    if all_token_results:
        df_all_t = pd.DataFrame(all_token_results)
        df_all_t.to_csv(os.path.join(WORKING_DIR, "MASTER_all_batches_tokens.csv"), index=False)
        print(f"✅ {len(df_all_t)} tokens → MASTER_all_batches_tokens.csv")

    total_q = sum(
        row.get('n_queries', 0)
        for row in all_batch_stats if row.get('status') == 'ok'
    )

    # =========================================================================
    # OVERALL SUMMARY TABLE
    # =========================================================================
    print(f"\n{'Method':<45} {'R@1':>7} {'R@5':>7} {'R@10':>7} "
          f"{'nDCG@1':>9} {'nDCG@5':>9} {'nDCG@10':>9}")
    print("-" * 105)

    for method in METHOD_KEYS:
        if method not in all_metrics:
            continue
        m   = all_metrics[method]
        cnt = m['count'] if m['count'] > 0 else 1
        print(f"{method:<45} "
              f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  "
              f"{m['n1']/cnt:8.4f}  {m['n5']/cnt:8.4f}  {m['n10']/cnt:8.4f}")

    # =========================================================================
    # PER-DOMAIN BREAKDOWN
    # =========================================================================
    print("\n" + "=" * 120)
    print("PER-DOMAIN BREAKDOWN")
    print("=" * 120)
    all_domains = sorted(all_domain_metrics.keys())

    for domain in all_domains:
        dm  = all_domain_metrics[domain]
        t_m = dm.get('traditional', _init_metric())
        cnt = t_m['count'] if t_m['count'] > 0 else 1
        print(f"\n  Domain: {domain}  (N={cnt})")
        print(f"  {'Method':<45} {'R@10':>7}  {'nDCG@10':>9}")
        print(f"  {'-'*65}")
        for method in METHOD_KEYS:
            m_  = dm.get(method, _init_metric())
            c_  = m_['count'] if m_['count'] > 0 else 1
            print(f"  {method:<45} {m_['r10']/c_*100:6.2f}%  {m_['n10']/c_:9.4f}")

    # =========================================================================
    # SAVE SUMMARIES
    # =========================================================================
    summary_rows = []
    for method in METHOD_KEYS:
        if method not in all_metrics:
            continue
        m   = all_metrics[method]
        cnt = m['count'] if m['count'] > 0 else 1
        summary_rows.append({
            'method':  method,
            'r@1':     round(m['r1']  / cnt * 100, 4),
            'r@5':     round(m['r5']  / cnt * 100, 4),
            'r@10':    round(m['r10'] / cnt * 100, 4),
            'ndcg@1':  round(m['n1']  / cnt,       6),
            'ndcg@5':  round(m['n5']  / cnt,       6),
            'ndcg@10': round(m['n10'] / cnt,       6),
        })
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_attn_pruning_summary.csv"), index=False
    )
    print("\n✅ MASTER_attn_pruning_summary.csv saved")

    if all_batch_stats:
        pd.DataFrame(all_batch_stats).to_csv(
            os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False
        )

except Exception as e:
    print(f"❌ Final aggregation error: {e}")
    import traceback; traceback.print_exc()


# ==============================================================================
# BƯỚC 5: VISUALIZE — 3 figures
# ==============================================================================
print("\n>>> BƯỚC 5: Visualize Attention Score Pruning Results")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import numpy as np

    METRICS_PLOT = [
        ('ndcg@10',   'n10', 'nDCG@10',   False),
        ('recall@10', 'r10', 'Recall@10',  True),
        ('ndcg@5',    'n5',  'nDCG@5',    False),
        ('recall@1',  'r1',  'Recall@1',   True),
    ]

    COLORS_N       = {1: '#1D9E75', 2: '#E07B39', 4: '#3B72C4', 8: '#9B59B6'}
    COLORS_SCORING = {'trad': '#1D9E75', 'weighted': '#E07B39'}
    COLOR_BASE     = '#888780'

    def _mval(key, mkey, pct):
        m   = all_metrics.get(key)
        if m is None or m['count'] == 0:
            return None
        v = m[mkey] / m['count']
        return v * 100 if pct else v

    # -----------------------------------------------------------------------
    # Figure 1: Main result — nDCG@10 & Recall@10 per n_last_layers
    #           (scoring = trad, paper-exact: n=1 highlighted)
    # -----------------------------------------------------------------------
    fig1, axes1 = plt.subplots(1, 2, figsize=(13, 5))

    for ai, (_, mkey, title, pct_flag) in enumerate(METRICS_PLOT[:2]):
        ax = axes1[ai]
        ax.set_title(f"Attention Score Pruning — {title}\n(scoring=trad, varying n_last_layers)",
                     fontsize=11)
        ax.set_xlabel('Top-k ratio (fraction of tokens kept)', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25, linewidth=0.6)
        ax.grid(axis='x', alpha=0.15, linewidth=0.4)
        ax.spines[['top', 'right']].set_visible(False)

        base_v = _mval('traditional', mkey, pct_flag)
        if base_v is not None:
            ax.axhline(base_v, color=COLOR_BASE, linewidth=1.8, linestyle='--',
                       label='Baseline (full ColBERT, no pruning)', zorder=2)

        for n_layers in N_LAST_LAYERS_LIST:
            vals = [_mval(f"attn_L{n_layers}_trad_r{int(r*100)}", mkey, pct_flag)
                    for r in TOPK_RATIOS]
            valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
            if not valid:
                continue
            xs, ys  = zip(*valid)
            lw      = 2.8 if n_layers == 1 else 1.8    # paper-exact gets thicker line
            label   = f"L={n_layers}" + (" ★ (paper-exact)" if n_layers == 1 else "")
            ax.plot(xs, ys, color=COLORS_N[n_layers], linewidth=lw,
                    marker='o', markersize=5, label=label, zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')

    plt.suptitle(
        "Token Pruning with Attention Score (Lassance et al. 2021)\n"
        "i(t) = Σ_h Σ_j (A)_{h,t,j}  — column-sum of last attention layer(s)",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    p1 = os.path.join(WORKING_DIR, "attn_pruning_main_nlayers.png")
    plt.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot 1 → {p1}")

    # -----------------------------------------------------------------------
    # Figure 2: Scoring mode comparison (trad vs weighted)
    #           Fixed n_last_layers = 1 (paper-exact)
    # -----------------------------------------------------------------------
    fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))

    for ai, (_, mkey, title, pct_flag) in enumerate(METRICS_PLOT[:2]):
        ax = axes2[ai]
        ax.set_title(f"Scoring Mode Comparison — {title}\n(n_last_layers=1, paper-exact)",
                     fontsize=11)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25, linewidth=0.6)
        ax.spines[['top', 'right']].set_visible(False)

        base_v = _mval('traditional', mkey, pct_flag)
        if base_v is not None:
            ax.axhline(base_v, color=COLOR_BASE, linewidth=1.8, linestyle='--',
                       label='Baseline (no pruning)', zorder=2)

        for scoring in SCORING_MODES:
            vals = [_mval(f"attn_L1_{scoring}_r{int(r*100)}", mkey, pct_flag)
                    for r in TOPK_RATIOS]
            valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
            if not valid:
                continue
            xs, ys = zip(*valid)
            label  = ('Traditional Sum' if scoring == 'trad'
                      else 'Importance-Weighted Sum')
            ax.plot(xs, ys, color=COLORS_SCORING[scoring], linewidth=2.2,
                    marker='o', markersize=5, label=label, zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=9, framealpha=0.4, loc='lower right')

    plt.suptitle(
        "Attention Score Pruning: Traditional vs Importance-Weighted Scoring\n"
        "(n_last_layers=1 — paper-exact setting)",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    p2 = os.path.join(WORKING_DIR, "attn_pruning_scoring_comparison.png")
    plt.savefig(p2, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot 2 → {p2}")

    # -----------------------------------------------------------------------
    # Figure 3: 4-panel overview (all 4 metrics), n_last_layers=1, trad scoring
    # -----------------------------------------------------------------------
    fig3, axes3 = plt.subplots(2, 2, figsize=(13, 9))
    axes3 = axes3.flatten()

    for ai, (_, mkey, title, pct_flag) in enumerate(METRICS_PLOT):
        ax = axes3[ai]
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25, linewidth=0.6)
        ax.grid(axis='x', alpha=0.15, linewidth=0.4)
        ax.spines[['top', 'right']].set_visible(False)

        base_v = _mval('traditional', mkey, pct_flag)
        if base_v is not None:
            ax.axhline(base_v, color=COLOR_BASE, linewidth=1.8, linestyle='--',
                       label='Baseline (full ColBERT)', zorder=2)

        vals = [_mval(f"attn_L1_trad_r{int(r*100)}", mkey, pct_flag) for r in TOPK_RATIOS]
        valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color='#1DB954', linewidth=2.2, marker='o', markersize=5,
                    label='Attention Score Pruning (n=1)', zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')

    plt.suptitle(
        "Token Pruning with Attention Score (Lassance et al. 2021) vs Baseline\n"
        "n_last_layers=1, scoring=traditional — paper-exact configuration",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    p3 = os.path.join(WORKING_DIR, "attn_pruning_4panel_overview.png")
    plt.savefig(p3, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot 3 → {p3}")

except Exception as e:
    print(f"⚠️  Plot error (non-fatal): {e}")
    import traceback; traceback.print_exc()

print("\n>>> BƯỚC 4+5 COMPLETE — ATTENTION SCORE PRUNING (Lassance et al. 2021) DONE")